<h2>Description</h2>

Dans ce code, nous allons établir un modèle afin de prédire le débit horaire sur les Champs Élysées.

Imports

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.inspection import permutation_importance

doc = 'champs_elysees.csv'

df_final = pd.read_csv('../datasets_axes_with_all_features/'+ doc, sep=';')

In [2]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9266 entries, 0 to 9265
Data columns (total 42 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Unnamed: 0                 9266 non-null   int64  
 1   Identifiant arc            9266 non-null   int64  
 2   Libelle                    9266 non-null   object 
 3   Date et heure de comptage  9266 non-null   object 
 4   Débit horaire              8703 non-null   float64
 5   Taux d'occupation          8644 non-null   float64
 6   Etat trafic                9266 non-null   object 
 7   Identifiant noeud amont    9266 non-null   int64  
 8   Libelle noeud amont        9266 non-null   object 
 9   Identifiant noeud aval     9266 non-null   int64  
 10  Libelle noeud aval         9266 non-null   object 
 11  Etat arc                   9266 non-null   object 
 12  Date debut dispo data      9266 non-null   object 
 13  Date fin dispo data        9266 non-null   objec

In [3]:
df_final = df_final.copy()
df_final['Date et heure de comptage'] = pd.to_datetime(df_final['Date et heure de comptage'], errors='coerce')
df_final = df_final.sort_values('Date et heure de comptage').reset_index(drop=True)

for col in ['est_vacances', 'est_ferie', 'est_avant_ferie', 'est_pieton']:
    if col in df_final.columns:
        df_final[col] = pd.to_numeric(df_final[col], errors='coerce')

features = [
    'Température', 'est_vacances', 'duree prec (en min)',
    'heure_sin', 'heure_cos', 'jour_sin', 'jour_cos', 'mois_sin', 'mois_cos',
    'force moyenne vent (m/s)', 'jour_semaine', 'mois',
    'est_ferie', 'est_avant_ferie', 'ensoleillement (en min)', 'est_pieton'
]
target = 'Débit horaire'

mask_known   = df_final[target].notna()
mask_missing = df_final[target].isna()

X_known = df_final.loc[mask_known, features].copy()
y_known = df_final.loc[mask_known, target].astype(float)
X_missing = df_final.loc[mask_missing, features].copy()

numeric_features = [
    'Température', 'est_vacances',
    'heure_sin', 'heure_cos', 'jour_sin', 'jour_cos', 'mois_sin', 'mois_cos',
    'force moyenne vent (m/s)', 'duree prec (en min)', 'mois',
    'est_ferie', 'est_avant_ferie', 'ensoleillement (en min)', 'est_pieton'
]
categorical_features = ['jour_semaine']

try:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)  
except TypeError:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)         

preprocess = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='median'))]), numeric_features),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', ohe)
        ]), categorical_features),
    ],
    remainder='drop'
)

model = HistGradientBoostingRegressor(
    loss='absolute_error',   
    max_depth=3,
    max_iter=400,
    early_stopping=False,
    random_state=42
)

pipe = Pipeline(steps=[('prep', preprocess), ('model', model)])

# ---------- 3) Split chronologique ----------
X_train, X_test, y_train, y_test = train_test_split(
    X_known, y_known, test_size=0.2, shuffle=False
)

# ---------- 4) Pondérations (férié / veille / piéton) ----------
W_FERIE   = 3.0
W_AVANT   = 1.5
W_PIETON  = 10
POST_SCALE_FERIE = 1.0  # laissez 1.0 si vous ne souhaitez pas corriger les prédictions les jours fériés

def make_weights(X_frame):
    w = np.ones(len(X_frame), dtype=float)
    is_ferie  = X_frame['est_ferie'].fillna(0).astype(int).to_numpy()
    is_avant  = X_frame['est_avant_ferie'].fillna(0).astype(int).to_numpy()
    is_pieton = X_frame['est_pieton'].fillna(0).astype(int).to_numpy()

    w[is_ferie == 1]  = W_FERIE
    w[is_avant == 1]  = np.maximum(w[is_avant == 1], W_AVANT)
    w[is_pieton == 1] = W_PIETON

    # Option : normalisation pour garder une échelle de perte comparable
    #w *= (len(w) / w.sum())
    return w

w_train = make_weights(X_train)

# ---------- 5) Entraînement ----------
pipe.fit(X_train, y_train, model__sample_weight=w_train)

# ---------- 6) Prédiction + post-ajustement éventuel ----------
y_pred = pipe.predict(X_test)

mask_ferie_test = X_test['est_ferie'].fillna(0).astype(int).to_numpy() == 1
mask_est_pieton_test = X_test['est_pieton'].fillna(0).astype(int).to_numpy() == 1
y_pred[mask_ferie_test] *= POST_SCALE_FERIE
#y_pred[mask_est_pieton_test] *= 0.5

# ---------- 7) Évaluation ----------
r2   = r2_score(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R² : {r2:.3f}")
print(f"MAE : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")

# Diagnostics par sous-régimes
is_pieton_test = X_test['est_pieton'].fillna(0).astype(int) == 1
print(f"Part d'observations piéton (test) : {is_pieton_test.mean():.1%}")
if is_pieton_test.any():
    mae_pieton = mean_absolute_error(y_test[is_pieton_test], y_pred[is_pieton_test])
    print(f"MAE (jours piéton) : {mae_pieton:.2f} (n={is_pieton_test.sum()})")
    mae_non_pieton = mean_absolute_error(y_test[~is_pieton_test], y_pred[~is_pieton_test])
    print(f"MAE (jours non piéton) : {mae_non_pieton:.2f} (n={(~is_pieton_test).sum()})")

# ---------- 8) Imputation des valeurs manquantes ----------
if mask_missing.any():
    df_final.loc[mask_missing, 'Débit_prédit'] = pipe.predict(X_missing)
    print(f"Imputation réalisée pour {mask_missing.sum()} lignes (colonne 'Débit_prédit').")


R² : 0.790
MAE : 85.00
RMSE : 116.24
Part d'observations piéton (test) : 1.8%
MAE (jours piéton) : 180.39 (n=31)
MAE (jours non piéton) : 83.27 (n=1710)
Imputation réalisée pour 563 lignes (colonne 'Débit_prédit').


In [4]:
from sklearn.inspection import permutation_importance
import pandas as pd
import numpy as np
import plotly.express as px

# 1) Importance par permutation sur le pipeline complet (prétraitements inclus)
perm = permutation_importance(
    estimator=pipe,
    X=X_test,
    y=y_test,
    n_repeats=20,
    random_state=42,
    scoring='neg_root_mean_squared_error'  # cohérent avec votre RMSE
)

imp_df = (
    pd.DataFrame({
        'feature': features,
        'importance_mean': perm.importances_mean,
        'importance_std': perm.importances_std
    })
    .sort_values('importance_mean', ascending=False)
)

print(imp_df.head(20))

# 2) Bar chart Plotly (top 20)
topk = imp_df.head(20).sort_values('importance_mean', ascending=True)
fig = px.bar(
    topk,
    x='importance_mean', y='feature',
    error_x='importance_std',
    orientation='h',
    title='Importance par permutation — Top 20 (plus haut = plus influent)'
)
fig.update_layout(xaxis_title="Perte de performance (Δ RMSE, signe inversé)", yaxis_title="")
fig.show()


                     feature  importance_mean  importance_std
3                  heure_sin       172.846978        4.248976
4                  heure_cos       116.734969        2.355896
5                   jour_sin        66.156411        2.467855
15                est_pieton        10.258142        1.314788
9   force moyenne vent (m/s)         6.112833        0.514907
6                   jour_cos         3.833528        0.464990
0                Température         3.236471        0.691256
10              jour_semaine         2.576663        0.391228
12                 est_ferie         2.183126        0.567914
14   ensoleillement (en min)         1.746609        0.327676
1               est_vacances         1.418166        0.381638
8                   mois_cos         0.104492        0.183567
13           est_avant_ferie         0.000000        0.000000
2        duree prec (en min)        -0.132946        0.161259
7                   mois_sin        -0.504831        0.411472
11      

In [5]:
time_index = df_final.loc[X_test.index, 'Date et heure de comptage']

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=time_index,
    y=y_pred,
    mode='lines',
    name='Débit prédit'
))
fig.add_trace(go.Scatter(
    x=time_index,
    y=y_test,
    mode='lines',
    name='Débit réel'
))

fig.update_layout(
    title="Comparaison des débits (réel vs prédit)",
    xaxis_title="Date et heure",
    yaxis_title="Débit horaire (véh/h)",
    hovermode='x unified'
)

fig.show()

In [6]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 0) Prédictions sur le jeu de test
y_pred = xgb.predict(X_val_t)

# 1) Reconstitution du DataFrame de diagnostic
n_test = len(y_test)
# récupère les n dernières dates (équivalent à df.loc[split:, 'Date et heure de comptage'])
dates_test = df['Date et heure de comptage'].iloc[-n_test:].values

diag = pd.DataFrame({
    'datetime': pd.to_datetime(dates_test),
    'y_true':   np.asarray(y_test),
    'y_pred':   np.asarray(y_pred)
}).sort_values('datetime').reset_index(drop=True)

diag['residual'] = diag['y_true'] - diag['y_pred']

# 2) Métriques globales
rmse = np.sqrt(mean_squared_error(diag['y_true'], diag['y_pred']))
mae  = mean_absolute_error(diag['y_true'], diag['y_pred'])
print(f"RMSE global : {rmse:,.2f}")
print(f"MAE  global : {mae:,.2f}")

# 3) Série temporelle Observé vs. Prédit
fig_ts = go.Figure()
fig_ts.add_trace(go.Scatter(x=diag['datetime'], y=diag['y_true'], mode='lines', name='Observé'))
fig_ts.add_trace(go.Scatter(x=diag['datetime'], y=diag['y_pred'], mode='lines', name='Prédit'))
fig_ts.update_layout(
    title="Débit horaire — Observé vs. Prédit (jeu de test)",
    xaxis_title="Date",
    yaxis_title="Débit horaire",
    legend_title_text="Série"
)
fig_ts.show()

# 4) Nuage de parité (y_true vs y_pred) + diagonale
min_v = float(np.min([diag['y_true'].min(), diag['y_pred'].min()]))
max_v = float(np.max([diag['y_true'].max(), diag['y_pred'].max()]))

fig_parity = go.Figure()
fig_parity.add_trace(go.Scatter(x=diag['y_true'], y=diag['y_pred'], mode='markers', name='Points', opacity=0.6))
fig_parity.add_trace(go.Scatter(x=[min_v, max_v], y=[min_v, max_v], mode='lines', name='y = x'))
fig_parity.update_layout(
    title="Parité — Observé vs. Prédit",
    xaxis_title="y_observé",
    yaxis_title="y_prédit"
)
fig_parity.show()

# 5) Résidus dans le temps
fig_res_time = go.Figure()
fig_res_time.add_trace(go.Scatter(x=diag['datetime'], y=diag['residual'], mode='lines', name='Résidu'))
fig_res_time.add_hline(y=0)
fig_res_time.update_layout(
    title="Résidus (y_observé − y_prédit) dans le temps",
    xaxis_title="Date",
    yaxis_title="Résidu"
)
fig_res_time.show()

# 6) Histogramme des résidus
fig_hist = px.histogram(diag, x='residual', nbins=40, title='Distribution des résidus')
fig_hist.update_layout(xaxis_title="Résidu", yaxis_title="Fréquence")
fig_hist.show()

# 7) RMSE glissant
window = 48  # ≈ 2 jours si série horaire sans trous
diag['se'] = (diag['y_true'] - diag['y_pred'])**2
diag['rmse_roll'] = np.sqrt(diag['se'].rolling(window=window, min_periods=window//2).mean())

fig_rmse_roll = go.Figure()
fig_rmse_roll.add_trace(go.Scatter(x=diag['datetime'], y=diag['rmse_roll'], mode='lines', name='RMSE glissant'))
fig_rmse_roll.update_layout(
    title=f"RMSE glissant (fenêtre = {window})",
    xaxis_title="Date",
    yaxis_title="RMSE"
)
fig_rmse_roll.show()

# 8) MAE par heure de la journée
tmp = diag.copy()
tmp['heure'] = pd.to_datetime(tmp['datetime']).dt.hour
mae_par_heure = tmp.groupby('heure').apply(lambda z: mean_absolute_error(z['y_true'], z['y_pred'])).reset_index(name='MAE')

fig_mae_hour = px.bar(mae_par_heure, x='heure', y='MAE', title='MAE par heure de la journée')
fig_mae_hour.update_layout(xaxis_title="Heure", yaxis_title="MAE")
fig_mae_hour.show()


NameError: name 'xgb' is not defined